In [1]:
import pandas as pd
import numpy as np
from pandas.api.types import CategoricalDtype
from collections import defaultdict

In [2]:
def preprocess(df):
    return df

df1 = preprocess(pd.read_parquet('train/5.잔액정보/201807_train_잔액정보.parquet'))
df2 = preprocess(pd.read_parquet('train/5.잔액정보/201808_train_잔액정보.parquet'))
df3 = preprocess(pd.read_parquet('train/5.잔액정보/201809_train_잔액정보.parquet'))
df4 = preprocess(pd.read_parquet('train/5.잔액정보/201810_train_잔액정보.parquet'))
df5 = preprocess(pd.read_parquet('train/5.잔액정보/201811_train_잔액정보.parquet'))
df6 = preprocess(pd.read_parquet('train/5.잔액정보/201812_train_잔액정보.parquet'))

In [3]:
dfs = [df.drop(columns=['기준년월'], errors='ignore') for df in [df1, df2, df3, df4, df5, df6]]

def merge_two_avg(df_left, df_right):
    merge_keys = ['ID']
    if 'Segment' in df_left.columns and 'Segment' in df_right.columns:
        merge_keys.append('Segment')

    merged = pd.merge(df_left, df_right, on=merge_keys, how='outer', suffixes=('_left', '_right'))
    result = merged[merge_keys].copy()
    
    # 평균 계산
    for col in set(df_left.columns).union(df_right.columns):
        if col in merge_keys:
            continue
        col_left = f"{col}_left" if f"{col}_left" in merged.columns else None
        col_right = f"{col}_right" if f"{col}_right" in merged.columns else None
        
        cols_to_avg = [c for c in [col_left, col_right] if c is not None]
        result[col] = merged[cols_to_avg].mean(axis=1, skipna=True)
    
    return result

from functools import reduce
merged_df = reduce(merge_two_avg, dfs)
merged_df

,ID,카드론잔액_최종경과월,RV_평균잔액_R3M,잔액_카드론_B0M,연체원금_B1M,평잔_CA_3M,RV_최대잔액_R12M,평잔_카드론_3M,평잔_CA_해외_6M,RV_최대잔액_R6M,...,평잔_할부_6M,연체잔액_할부_해외_B0M,월중평잔_할부_B0M,연체잔액_RV일시불_B0M,잔액_리볼빙일시불이월_B0M,월중평잔_할부,평잔_카드론_6M,평잔_CA_6M,잔액_현금서비스_B0M,연체잔액_일시불_해외_B0M
0,TRAIN_000000,0.0,0.0000,0.00000,0.0,25588.81250,0.00000,0.00000,0.0,0.000,...,615.21875,0.0,525.06250,0.0,0.00000,503.00000,0.00000,20433.7500,24153.56250,0.0
1,TRAIN_000001,0.0,0.0000,0.00000,0.0,0.00000,0.00000,0.00000,0.0,0.000,...,2183.18750,0.0,1598.96875,0.0,0.00000,1539.75000,0.00000,0.0000,0.00000,0.0
2,TRAIN_000002,0.0,2951.3125,0.00000,0.0,48510.75000,4920.28125,0.00000,0.0,5490.375,...,6002.25000,0.0,3302.87500,0.0,3530.53125,4064.87500,0.00000,55016.6875,22382.28125,0.0
3,TRAIN_000003,0.0,0.0000,0.00000,0.0,22501.90625,0.00000,0.00000,0.0,0.000,...,3956.87500,0.0,2391.71875,0.0,0.00000,2186.93750,0.00000,18162.1875,25573.09375,0.0
4,TRAIN_000004,0.0,0.0000,0.00000,0.0,0.00000,0.00000,0.00000,0.0,0.000,...,0.00000,0.0,0.00000,0.0,0.00000,0.00000,0.00000,0.0000,0.00000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399995,TRAIN_399995,0.0,0.0000,0.00000,0.0,0.00000,0.00000,0.00000,0.0,0.000,...,0.00000,0.0,0.00000,0.0,0.00000,0.00000,0.00000,0.0000,0.00000,0.0
399996,TRAIN_399996,0.0,0.0000,28906.65625,0.0,0.00000,0.00000,18666.21875,0.0,0.000,...,51.18750,0.0,0.00000,0.0,0.00000,0.00000,23760.78125,0.0000,0.00000,0.0
399997,TRAIN_399997,0.0,0.0000,0.00000,0.0,0.00000,0.00000,0.00000,0.0,0.000,...,3514.62500,0.0,3121.21875,0.0,0.00000,2958.21875,0.00000,0.0000,0.00000,0.0
399998,TRAIN_399998,0.0,0.0000,0.00000,0.0,0.00000,0.00000,0.00000,0.0,0.000,...,0.00000,0.0,0.00000,0.0,0.00000,0.00000,0.00000,0.0000,0.00000,0.0


In [4]:
df_segment = pd.read_parquet('train/1.회원정보/201807_train_회원정보.parquet')[['ID', 'Segment']]

# 2. 중복 제거 (ID별 Segment가 유일하다는 전제)
df_segment = df_segment.drop_duplicates(subset='ID')

# 3. merged_df에 Segment 열 붙이기 (ID 기준)
merged_df = pd.merge(merged_df, df_segment, on='ID', how='left')
merged_df

,ID,카드론잔액_최종경과월,RV_평균잔액_R3M,잔액_카드론_B0M,연체원금_B1M,평잔_CA_3M,RV_최대잔액_R12M,평잔_카드론_3M,평잔_CA_해외_6M,RV_최대잔액_R6M,...,연체잔액_할부_해외_B0M,월중평잔_할부_B0M,연체잔액_RV일시불_B0M,잔액_리볼빙일시불이월_B0M,월중평잔_할부,평잔_카드론_6M,평잔_CA_6M,잔액_현금서비스_B0M,연체잔액_일시불_해외_B0M,Segment
0,TRAIN_000000,0.0,0.0000,0.00000,0.0,25588.81250,0.00000,0.00000,0.0,0.000,...,0.0,525.06250,0.0,0.00000,503.00000,0.00000,20433.7500,24153.56250,0.0,D
1,TRAIN_000001,0.0,0.0000,0.00000,0.0,0.00000,0.00000,0.00000,0.0,0.000,...,0.0,1598.96875,0.0,0.00000,1539.75000,0.00000,0.0000,0.00000,0.0,E
2,TRAIN_000002,0.0,2951.3125,0.00000,0.0,48510.75000,4920.28125,0.00000,0.0,5490.375,...,0.0,3302.87500,0.0,3530.53125,4064.87500,0.00000,55016.6875,22382.28125,0.0,C
3,TRAIN_000003,0.0,0.0000,0.00000,0.0,22501.90625,0.00000,0.00000,0.0,0.000,...,0.0,2391.71875,0.0,0.00000,2186.93750,0.00000,18162.1875,25573.09375,0.0,D
4,TRAIN_000004,0.0,0.0000,0.00000,0.0,0.00000,0.00000,0.00000,0.0,0.000,...,0.0,0.00000,0.0,0.00000,0.00000,0.00000,0.0000,0.00000,0.0,E
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399995,TRAIN_399995,0.0,0.0000,0.00000,0.0,0.00000,0.00000,0.00000,0.0,0.000,...,0.0,0.00000,0.0,0.00000,0.00000,0.00000,0.0000,0.00000,0.0,E
399996,TRAIN_399996,0.0,0.0000,28906.65625,0.0,0.00000,0.00000,18666.21875,0.0,0.000,...,0.0,0.00000,0.0,0.00000,0.00000,23760.78125,0.0000,0.00000,0.0,D
399997,TRAIN_399997,0.0,0.0000,0.00000,0.0,0.00000,0.00000,0.00000,0.0,0.000,...,0.0,3121.21875,0.0,0.00000,2958.21875,0.00000,0.0000,0.00000,0.0,C
399998,TRAIN_399998,0.0,0.0000,0.00000,0.0,0.00000,0.00000,0.00000,0.0,0.000,...,0.0,0.00000,0.0,0.00000,0.00000,0.00000,0.0000,0.00000,0.0,E


In [5]:
nan_columns = merged_df.columns[merged_df.isnull().any()].tolist()

print("NaN이 포함된 열 목록:")
print(nan_columns)

NaN이 포함된 열 목록:
['연체일자_B0M']


In [6]:
ex1 = merged_df

In [7]:
cols_to_drop = ['연체일자_B0M']
ex1.drop(columns=cols_to_drop, inplace=True)

In [8]:
missing_mask = ex1.isna() | (ex1 == -1)
missing_ratio = missing_mask.mean()

high_na = missing_ratio[missing_ratio > 0.2].index.tolist()

high_const_cols = []
threshold_const = 0.8

for col in ex1.columns:
    top_ratio = ex1[col].value_counts(normalize=True, dropna=False).values[0]
    if top_ratio > threshold_const:
        high_const_cols.append(col)

to_drop = list(set(high_na + high_const_cols))

if 'Segment' in to_drop:
    to_drop.remove('Segment')

print("삭제 대상 컬럼 (결측>20% 또는 동일값>80%):", to_drop)

ex1.drop(columns=to_drop, inplace=True)

삭제 대상 컬럼 (결측>20% 또는 동일값>80%): ['카드론잔액_최종경과월', 'RV_평균잔액_R3M', '잔액_카드론_B0M', 'RV잔액이월횟수_R3M', '연체원금_B1M', '연체일수_B1M', '평잔_할부_해외_6M', '평잔_CA_3M', 'RV_최대잔액_R12M', '잔액_할부_해외_B0M', '평잔_일시불_해외_3M', '평잔_카드론_3M', '잔액_카드론_B4M', '평잔_CA_해외_3M', '평잔_일시불_해외_6M', '평잔_CA_해외_6M', '평잔_RV일시불_6M', 'RV_최대잔액_R6M', '연체잔액_B0M', '연체잔액_RV일시불_해외_B0M', '연체잔액_일시불_B0M', '연체잔액_카드론_B0M', '잔액_현금서비스_B1M', '잔액_카드론_B1M', '연체일수_최근', 'RV잔액이월횟수_R6M', '잔액_카드론_B3M', '최종연체개월수_R15M', '연체원금_최근', '연체일수_B2M', '연체잔액_CA_B0M', '잔액_리볼빙CA이월_B0M', '평잔_RV일시불_해외_3M', '연체잔액_CA_해외_B0M', '연체잔액_현금서비스_B0M', '평잔_카드론_6M', '매각잔액_B1M', 'RV_평균잔액_R6M', '평잔_할부_해외_3M', '월중평잔_카드론', '연체잔액_대환론_B0M', '연체원금_B2M', '잔액_할부_유이자_B0M', '월중평잔_CA_B0M', '평잔_RV일시불_해외_6M', 'RV_최대잔액_R3M', '월중평잔_CA', '연체잔액_할부_B0M', '잔액_카드론_B5M', 'RV_평균잔액_R12M', '잔액_카드론_B2M', '잔액_현금서비스_B2M', '연체잔액_할부_해외_B0M', '연체잔액_RV일시불_B0M', '잔액_리볼빙일시불이월_B0M', '월중평잔_RV일시불', '평잔_CA_6M', '잔액_현금서비스_B0M', '평잔_RV일시불_3M', '연체잔액_일시불_해외_B0M']


In [9]:
num_df = ex1.select_dtypes(include=[np.number]).dropna()

corr = num_df.corr().abs()

high_corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
)
high_corr_pairs.columns = ['Feature_1', 'Feature_2', 'Correlation']
high_corr_pairs = high_corr_pairs[high_corr_pairs['Correlation'] > 0.8]

if not np.issubdtype(ex1['Segment'].dtype, np.number):
    segment_map = {label: idx for idx, label in enumerate(sorted(ex1['Segment'].unique()))}
    ex1['Segment_encoded'] = ex1['Segment'].map(segment_map)
else:
    ex1['Segment_encoded'] = ex1['Segment']

segment_corr = ex1[num_df.columns].corrwith(ex1['Segment_encoded']).abs()

high_corr_pairs['Corr_with_Segment_1'] = high_corr_pairs['Feature_1'].map(segment_corr)
high_corr_pairs['Corr_with_Segment_2'] = high_corr_pairs['Feature_2'].map(segment_corr)

high_corr_pairs = high_corr_pairs.sort_values(by='Correlation', ascending=False).reset_index(drop=True)

print(f"▶ 상관계수 0.7 초과 변수쌍 수: {len(high_corr_pairs)}")
display(high_corr_pairs)


▶ 상관계수 0.7 초과 변수쌍 수: 52


,Feature_1,Feature_2,Correlation,Corr_with_Segment_1,Corr_with_Segment_2
0,월중평잔_할부_B0M,월중평잔_할부,0.995940,0.249834,0.247557
1,잔액_할부_B1M,월중평잔_할부_B0M,0.994344,0.244665,0.249834
2,잔액_할부_B1M,월중평잔_할부,0.991996,0.244665,0.247557
3,잔액_할부_B1M,잔액_할부_B2M,0.990730,0.244665,0.242546
4,잔액_할부_B1M,평잔_할부_3M,0.986086,0.244665,0.245363
5,잔액_할부_B2M,월중평잔_할부_B0M,0.984687,0.242546,0.249834
6,잔액_할부_B2M,평잔_할부_3M,0.984118,0.242546,0.245363
7,평잔_할부_3M,월중평잔_할부_B0M,0.983526,0.245363,0.249834
8,잔액_할부_B2M,월중평잔_할부,0.983188,0.242546,0.247557
9,평잔_할부_3M,월중평잔_할부,0.982918,0.245363,0.247557


In [10]:
to_drop = []

for _, row in high_corr_pairs.iterrows():
    f1, f2 = row['Feature_1'], row['Feature_2']
    c1, c2 = row['Corr_with_Segment_1'], row['Corr_with_Segment_2']
    
    if pd.isna(c1) or pd.isna(c2):
        continue
    
    if c1 < c2:
        to_drop.append(f1)
    else:
        to_drop.append(f2)

to_drop = list(set(to_drop))

# 결과 출력
print(f"▶ 제거 대상 피처 수: {len(to_drop)}")
print("제거할 피처 목록:")
print(to_drop)

▶ 제거 대상 피처 수: 15
제거할 피처 목록:
['평잔_6M', '잔액_할부_무이자_B0M', '평잔_할부_3M', '월중평잔_일시불', '평잔_3M', '월중평잔_일시불_B0M', '평잔_할부_6M', '월중평잔_할부_B0M', '잔액_일시불_B2M', '월중평잔_할부', '잔액_할부_B2M', '평잔_일시불_3M', '잔액_할부_B1M', '잔액_일시불_B0M', '잔액_일시불_B1M']


In [11]:
cols_to_drop = ['평잔_6M', '잔액_할부_무이자_B0M', '평잔_할부_3M', '월중평잔_일시불', '평잔_3M', '월중평잔_일시불_B0M', '평잔_할부_6M', '월중평잔_할부_B0M', '잔액_일시불_B2M', '월중평잔_할부', '잔액_할부_B2M', '평잔_일시불_3M', '잔액_할부_B1M', '잔액_일시불_B0M', '잔액_일시불_B1M']
ex1.drop(columns=cols_to_drop, inplace=True)

In [12]:
cols_to_drop = ['Segment_encoded']
ex1.drop(columns=cols_to_drop, inplace=True)
cols = ex1.columns.tolist()
cols

['ID', '최종연체회차', '월중평잔', '잔액_할부_B0M', '평잔_일시불_6M', 'Segment']

In [21]:
ex1.to_parquet('잔액_전처리_Segment.parquet', index=False)

In [14]:
ddf1 = preprocess(pd.read_parquet('train/5.잔액정보/201807_train_잔액정보.parquet'))
ddf2 = preprocess(pd.read_parquet('train/5.잔액정보/201808_train_잔액정보.parquet'))
ddf3 = preprocess(pd.read_parquet('train/5.잔액정보/201809_train_잔액정보.parquet'))
ddf4 = preprocess(pd.read_parquet('train/5.잔액정보/201810_train_잔액정보.parquet'))
ddf5 = preprocess(pd.read_parquet('train/5.잔액정보/201811_train_잔액정보.parquet'))
ddf6 = preprocess(pd.read_parquet('train/5.잔액정보/201812_train_잔액정보.parquet'))

In [15]:

dfs = [ddf1, ddf2, ddf3, ddf4, ddf5, ddf6]
for i in range(len(dfs)):
    if '기준년월' in dfs[i].columns:
        dfs[i] = dfs[i].drop(columns=['기준년월'])

# ID 기준으로 병합 후 평균
from functools import reduce

merged_df = reduce(
    lambda left, right: pd.merge(left, right, on='ID', how='outer', suffixes=('', '_dup')),
    dfs
)

# 같은 이름의 열 평균 구하기
from collections import defaultdict
import pandas as pd

result = pd.DataFrame()
result['ID'] = merged_df['ID']

# 열 이름 모아 평균 구하기
col_dict = defaultdict(list)
for col in merged_df.columns:
    if col != 'ID':
        base_col = col.split('_dup')[0]
        col_dict[base_col].append(col)

for base_col, cols in col_dict.items():
    result[base_col] = merged_df[cols].mean(axis=1, skipna=True)

# 결과 확인
print(result.head())

             ID   잔액_일시불_B0M    잔액_할부_B0M  잔액_현금서비스_B0M  잔액_리볼빙일시불이월_B0M  \
0  TRAIN_000000   733.000000   892.576923  24287.153846         0.000000   
1  TRAIN_000001  2560.576923  1804.615385      0.000000         0.000000   
2  TRAIN_000002  5972.192308  2935.115385  22999.076923      3876.923077   
3  TRAIN_000003  1250.384615  3474.038462  25051.500000         0.000000   
4  TRAIN_000004     0.000000     0.000000      0.000000         0.000000   

   잔액_리볼빙CA이월_B0M  잔액_카드론_B0M  월중평잔_일시불_B0M  월중평잔_할부_B0M   월중평잔_CA_B0M  ...  \
0             0.0         0.0    911.500000   552.384615  27404.500000  ...   
1             0.0         0.0   2630.769231  1834.538462      0.000000  ...   
2             0.0         0.0   6457.923077  3806.653846  28015.192308  ...   
3             0.0         0.0    884.692308  2758.153846  29815.769231  ...   
4             0.0         0.0      0.000000     0.000000      0.000000  ...   

          평잔_6M    평잔_일시불_6M  평잔_일시불_해외_6M  평잔_RV일시불_6M  평잔_RV일시불_해외

In [17]:
cols = ['ID', '최종연체회차', '월중평잔', '잔액_할부_B0M', '평잔_일시불_6M']

result = result[cols]
result

,ID,최종연체회차,월중평잔,잔액_할부_B0M,평잔_일시불_6M
0,TRAIN_000000,0.0,17876.615385,892.576923,2548.076923
1,TRAIN_000001,0.0,5420.461538,1804.615385,2772.769231
2,TRAIN_000002,0.0,47575.269231,2935.115385,8808.576923
3,TRAIN_000003,-99.0,32178.230769,3474.038462,1686.307692
4,TRAIN_000004,-99.0,84.423077,0.000000,13.500000
...,...,...,...,...,...
399995,TRAIN_399995,-99.0,0.000000,0.000000,0.000000
399996,TRAIN_399996,-99.0,35730.923077,0.000000,13256.961538
399997,TRAIN_399997,-99.0,8524.807692,2282.884615,3253.846154
399998,TRAIN_399998,-99.0,0.000000,0.000000,0.000000


In [20]:
result.to_parquet('잔액_전처리_test.parquet', index=False)

In [19]:
nan_columns = result.columns[result.isnull().any()].tolist()

print("NaN이 포함된 열 목록:")
print(nan_columns)

NaN이 포함된 열 목록:
[]
